## Model loading 

In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

### important !!!!!!!           # cVoice13_tone_v2Mapping   cVoice13_2nd_withDiph
path = "/Users/evanchan19/Desktop/Thesis_Research/Mostafa_Phono_model/models/cVoice13_IpaAndTone_2nd"
processor = Wav2Vec2Processor.from_pretrained(path)
model = Wav2Vec2ForCTC.from_pretrained(path)

# 1) Load the datasets and Parameters Settings

In [ ]:
from datasets import load_from_disk
import pandas as pd
import torch

data_path = "/Users/evanchan19/Desktop/Thesis_Research/Mostafa_Phono_model/speech_attribute/datasets/latic_l2_v1"

## Initial settings   ###  Importantant                         STEP 1)
file_path = "/Users/evanchan19/Desktop/Thesis_Research/Mostafa_Phono_model/speech_attribute/data/Mandarin_List_attr_IPAwithTone_V2.txt"

# Load the attribute mapping file                               STEP 2)
csv_file_path = "/Users/evanchan19/Desktop/Thesis_Research/Mostafa_Phono_model/speech_attribute/data/IPAwithTone_p2attr_V2.csv"
attribute_mapping = pd.read_csv(csv_file_path)

#### !!!!! VERY IMPORTANT                                       STEP 3)
which_type = "transcript_IPAwithTone_actual"  # This is from the datasets itself !!!
which_sentence = "sentence_speaker_said"  # This is from the datasets itself !!!
suppose_sentence = "transcript_IPAwithTone_suppose"

### Chocie can be                   'tone'     or     'IPA'     STEP 4)
choice = "IpaAndTone"

data = load_from_disk(data_path)
data = data["train"]
data = data.select(range(50))  # Select the first 1000 samples for testing
data

### 2) Function settings

In [ ]:
import re


## Step 1
def load_attribute_list(file_path):
    try:
        with open(file_path) as f:
            list_att = f.read().splitlines()
            return list_att
    except FileNotFoundError:
        raise FileNotFoundError(f"The list of attribute file {file_path} is not found")


list_att = load_attribute_list(file_path)


## Step 2
def create_binary_groups(list_att):
    groups = []
    for att in list_att:
        binary_att = [f"p_{att}", f"n_{att}"]  # Each attribute could be +ve or -ve
        groups.append(binary_att)
    return groups


groups = create_binary_groups(list_att)


## Step 3
def get_att_group_indx_map(bTraining=True):
    # Get group ids dictionary
    group_ids = [sorted(processor.tokenizer.convert_tokens_to_ids(group)) for group in groups]
    if bTraining:
        group_ids = [dict([(x[1], x[0] + 1) for x in list(enumerate(g))]) for g in group_ids]
    else:
        group_ids = [dict([(x[0] + 1, x[1]) for x in list(enumerate(g))]) for g in group_ids]
    return group_ids


group_ids = get_att_group_indx_map(bTraining=False)


## Step 4) Decouple if needed
decouple_diph_file = "/Users/evanchan19/Desktop/Thesis_Research/Mostafa_Phono_model/speech_attribute/data/Diphthongs_Mandarin.csv"
with open(decouple_diph_file, "r", encoding="utf-8") as f:
    diphthongs_to_monophthongs_map = dict([(x.split(",")[0], " ".join(x.split(",")[1:])) for x in f.read().splitlines()])


def decouple_diphthongs(batch):
    # Compile a regular expression pattern to match any diphthong in the map
    pattern = r"|".join(re.escape(diph) for diph in diphthongs_to_monophthongs_map.keys())

    def replace_diphthong(phoneme_string):
        return re.sub(pattern, lambda x: diphthongs_to_monophthongs_map[x.group(0)], phoneme_string).split()

    return [phoneme for phoneme_string in batch for phoneme in replace_diphthong(phoneme_string)]

## 3) Parameters, mode, file pre settings

In [ ]:
def transpose_matrix(matrix):
    # Transpose the list of lists
    transposed_matrix = list(zip(*matrix))
    return transposed_matrix


def print_vertical(matrix):
    # Print each row of the matrix in a vertical format
    for row in matrix:
        print(" | ".join(row))


# Assuming `data` is a list of dictionaries with audio and reference sentences
prediction = []  ## A ordered speech attribute mapping of predcitions !!!!!!!
actual_sentence = []  ## A actual_sentence for checking !!!!!!!!!!
supposed_sentence = []
Mandarin_sentence = []
for item in data:
    audio_array = item["audio"]["array"]  # Extract the audio array
    sampling_rate = item["audio"]["sampling_rate"]  # Extract the sampling rate

    # Process the audio
    inputs = processor(audio_array, sampling_rate=sampling_rate, return_tensors="pt", padding=True)

    # Run the model inference
    with torch.no_grad():
        logits = model(inputs.input_values).logits

    item_output = []  # Store pred_temps for the current item only
    for i in range(len(group_ids)):
        mask = torch.zeros(logits.size()[2], dtype=torch.bool)
        mask[0] = True
        mask[list(group_ids[i].values())] = True
        logits_g = logits[:, :, mask]
        pred_ids = torch.argmax(logits_g, dim=-1)
        pred_ids = pred_ids.cpu().apply_(lambda x: group_ids[i].get(x, x))

        # Decode predictions and clean up
        temp = processor.batch_decode(pred_ids, spaces_between_special_tokens=True)[0]
        temp = temp.replace("p_", "")  # Perform replace during the temp stage
        item_output.append(temp.split())  # Collect the prediction parts for the current item

    # Append the item's transposed predictions to the global prediction
    transposed_item_output = transpose_matrix(item_output)
    prediction.append(transposed_item_output)  # Keep accumulating all items’ predictions in prediction

    actual_sentence.append(item[which_type].split())
    supposed_sentence.append(item[suppose_sentence])
    Mandarin_sentence.append(item[which_sentence])  # this is Mandarin word

    # Print actual_sentence and prediction
    # print("-" * 100)
    # print("Actual Sentence:", item[suppose_sentence])
    # print("Actual correct IPA phoneme/ tone:", item[which_type])
    # print("Model Prediction from what Speakers said:")
    # print_vertical(transposed_item_output)


if "tone" in which_type:
    actual_sentence = [[int(item) for item in sublist] for sublist in actual_sentence]

# Important
# is_decouple = True
# if (is_decouple):
#     actual_sentence = [decouple_diphthongs(item) for item in actual_sentence]

# Pronunication Feedback Early design

In [ ]:
from Levenshtein import editops
from collections import defaultdict
from typing import List, Dict, Tuple
import pandas as pd

# Function to compute similarity between two sets of attributes
method = "jaccard"


def compute_similarity(set_a: set, set_b: set, method: str) -> float:
    if not set_a and not set_b:
        return 1.0  # Both empty sets are considered fully similar

    intersection = len(set_a & set_b)

    if method == "dice":
        total = len(set_a) + len(set_b)
        return 2 * intersection / total if total != 0 else 0.0
    elif method == "jaccard":
        union = len(set_a | set_b)
        return intersection / union if union != 0 else 0.0
    else:
        raise ValueError(f"Unsupported similarity method: {method}")


def attribute_matching(csv_dataFrame: pd.DataFrame, recognized_attribute_list: List[str], choice: str, method: str) -> Tuple[str, bool]:
    if csv_dataFrame.empty or not recognized_attribute_list:
        return "NoMandarinAttributeDetected!", True

    best_match = "NoMandarinAttributeDetected!"
    best_score = 0.0
    used_similarity = True

    recognized_set = set(recognized_attribute_list)

    for _, row in csv_dataFrame.iterrows():
        expected_set = {attr for attr, val in row.items() if val == 1}

        # Scenario 1: Exact match
        if recognized_set == expected_set:
            return row[choice], False

        # Scenario 2: Similarity match
        score = compute_similarity(recognized_set, expected_set, method=method)

        if score > best_score:
            best_score = score
            best_match = row[choice]

    return best_match, used_similarity


# Only useful in IpaAndTone together
def detail_checking(reference, predicted):
    result = {"phoneme_errors": {}, "tone_errors": {}}

    if not reference or not predicted:
        return result

    pred_phoneme = predicted[:-1] if predicted and predicted[-1].isdigit() else predicted
    pred_tone = predicted[-1] if predicted and predicted[-1].isdigit() else None
    rec_phoneme = reference[:-1] if reference and reference[-1].isdigit() else reference
    rec_tone = reference[-1] if reference and reference[-1].isdigit() else None

    if pred_phoneme != rec_phoneme:
        result["phoneme_errors"][rec_phoneme] = result["phoneme_errors"].get(rec_phoneme, 0) + 1
    elif pred_tone != rec_tone:
        result["tone_errors"][rec_tone] = result["tone_errors"].get(rec_tone, 0) + 1

    return result


# Compute the TA,FA,TR,FR
# Three sentence are needed for computing TA,FA,TR,FR
def compute_metrics(prediction: List, actual: List, supposed: List) -> Tuple[List[str], List[str]]:
    pred = prediction.split()
    actual = actual.split()
    supposed = supposed.split()

    # Metrcis_result only Store For TA,FA,TR,FR
    metrics_result = []
    diagnosed_result = []

    for i in range(len(pred)):
        temp = ""

        # Compute Accpetance and Rejection for the three list comparsion
        temp0 = "A" if actual[i] == supposed[i] else "R"
        temp1 = "A" if pred[i] == supposed[i] else "R"

        # Final Compute the TA,FA,TR,FR
        temp += "T" if temp0 == temp1 else "F"
        temp += "A" if pred[i] == supposed[i] else "R"

        metrics_result.append(temp)

        diagonsed = "CD" if pred[i] == actual[i] else "WD"
        diagnosed_result.append(diagonsed)

    return metrics_result, diagnosed_result


def compute_metrics_percentage(metrics: List[str], diagnosed: List[str]) -> Tuple[float, float, float, float, float, float, float, float]:
    total = len(metrics)
    TA = metrics.count("TA")
    TR = metrics.count("TR")
    FA = metrics.count("FA")
    FR = metrics.count("FR")
    CD = diagnosed.count("CD")
    WD = diagnosed.count("WD")

    # Compute percentages
    TA_percent = round(TA / total, 2) * 100
    TR_percent = round(TR / total, 2) * 100
    FA_percent = round(FA / total, 2) * 100
    FR_percent = round(FR / total, 2) * 100
    CD_percent = round(CD / total, 2) * 100
    WD_percent = round(WD / total, 2) * 100

    # Compute rates
    FAR = round(FA / (FA + TR), 2) * 100 if (FA + TR) > 0 else 0.0
    FRR = round(FR / (FR + TA), 2) * 100 if (FR + TA) > 0 else 0.0
    DER = round(WD / (CD + WD), 2) * 100 if (CD + WD) > 0 else 0.0

    return TA_percent, TR_percent, FA_percent, FR_percent, CD_percent, WD_percent, FAR, FRR, DER


# This is for one-to-one matching only
def compute_error_type(prediction: List, supposed: List):
    pred_temp = prediction.split()
    supposed_temp = supposed.split()
    result = []

    for i in range(len(pred_temp)):
        if pred_temp[i] == supposed_temp[i]:
            result.append("ok")
            continue

        if pred_temp[i] == "Unknown":
            result.append("Not CN word")
            continue

        if len(pred_temp[i]) == len(supposed_temp[i]):
            result.append("Substitution")
        elif len(pred_temp[i]) > len(supposed_temp[i]):
            result.append("Insertion")
        else:
            result.append("Deletion")

    return result


# This is for non-one-to-one matching
def compute_levenshtein_metrics(ref: List[str], pred: List[str]) -> Tuple[Dict, List[str]]:
    cm = defaultdict(lambda: defaultdict(int))
    ops = editops(ref, pred)

    # 每个 index insert，在前面插入时不能直接操作原始列表
    error_map = defaultdict(list)

    for op, ref_i, pred_i in ops:
        if op == "insert":
            msg = f"ins({pred_i}:'{pred[pred_i]}')"
            error_map[ref_i].append(msg)
            cm["insertions"][pred[pred_i]] += 1
        elif op == "delete":
            msg = f"del({ref_i}:'{ref[ref_i]}')"
            error_map[ref_i].append(msg)
            cm["deletions"][ref[ref_i]] += 1
        elif op == "replace":
            msg = f"sub({ref_i}:'{ref[ref_i]}' -> '{pred[pred_i]}')"
            error_map[ref_i].append(msg)
            cm["substitutions"][(ref[ref_i], pred[pred_i])] += 1

    # 构造 error_list：每个 token 按照 ref 的顺序走，如果某处有插入，也加进去
    error_list = []
    for i in range(len(ref)):
        if i in error_map:
            for msg in error_map[i]:
                if msg.startswith("ins"):
                    error_list.append(msg)
        if i in error_map and any(m.startswith("del") or m.startswith("sub") for m in error_map[i]):
            # 如果有 delete 或 substitution，优先取第一个非-insert的
            main_error = next(m for m in error_map[i] if not m.startswith("ins"))
            error_list.append(main_error)
        else:
            error_list.append("ok")

    # 补充末尾如果还有 insert 操作（例如插入到 ref 最末尾）
    if len(ref) in error_map:
        for msg in error_map[len(ref)]:
            if msg.startswith("ins"):
                error_list.append(msg)

    return cm, error_list


def pretty_print_leveshtein_errors(errors: List[str]):
    for error in errors:
        print(error)


def align_sentence(ref: List[str], target: List[str], errors: List[str]):
    # Align the reference and prediction with error tags
    aligned_ref = []
    aligned_target = []

    ref_index = 0
    index = 0  # Track position in reference

    for error in errors:
        if error.startswith("ins"):  # Insertion case
            aligned_ref.append("ins")  # Add the phoneme from reference
            ref_index += 1
            aligned_target.append("emp")

            if index + 1 == len(target):
                if errors[-1].startswith("del"):
                    pass
                else:
                    aligned_target.append(target[index])

            if index < len(target) and index + 1 != len(target):
                aligned_target.append(target[index])

        elif error.startswith("del"):  # Deletion case

            # Special case when Actual sentence second time to align with supposed
            if index < len(target):
                if target[index] == "emp":
                    aligned_ref.append("emp")
                else:
                    aligned_ref.append("del")  # Placeholder for deletion

            if index < len(target):
                aligned_target.append(target[index])

        else:
            aligned_ref.append(ref[ref_index])
            ref_index += 1

            if index < len(target):
                aligned_target.append(target[index])

        index += 1  # Move to the next reference element

    return " ".join(aligned_ref), " ".join(aligned_target)


def check_prediction(predicted_attributes, reference, choice, detailCheck: bool):
    # step 1) First check the choice of prediction
    if choice == "tone":
        name = "Phoneme_PinyinTon"
    elif choice == "IPA":
        name = "Phoneme_ipaDragon"
    elif choice == "IpaAndTone":
        name = "Phoneme_ipaDragon"
    else:
        return "wrong function input"

    # Then Filter the mapping for the current phoneme tone
    expected_attributes = attribute_mapping[attribute_mapping[name] == reference]
    ## Optional -> Show the refernce in Pinyin/tone
    # reference = expected_attributes[optional].iloc[0]

    expected_attributes = expected_attributes.iloc[0].iloc[3:]

    # Step 2) Run the checking
    expected_attributes = [attribute for attribute, expected in expected_attributes.items() if expected == 1]

    predicted_set = set(predicted_attributes)
    correct_set = set(expected_attributes)

    check = correct_set.issubset(predicted_set)
    wrong_list = list(correct_set - predicted_set)
    extra_list = list(predicted_set - correct_set)

    # Step 3) Convert the prediction into Phoneme/tone
    recognized_phoneme, is_similiar = attribute_matching(attribute_mapping, predicted_attributes, name, method)

    if check:
        print(f"✅ Correct Pronunciation!")
    elif check and is_similiar:
        print(f"✅ Similar Pronunciation Detected!")
    else:
        print("❌ Wrong Pronunciation Detected!")

    print(f"Phoneme/Tone: {reference}")

    # Print the most similar pronunciation if available
    if is_similiar:
        print(f"Most Similar Pronunciation: {recognized_phoneme}\n")
    else:
        print(f"Recognized as: {recognized_phoneme}\n")

    # Step 4: Print detailed attribute analysis if requested
    if detailCheck and is_similiar or not check:
        print("Details:")
        print(f"  - Missing Attributes: {', '.join(wrong_list) if wrong_list else 'None'}")
        print(f"  - Extra Attributes: {', '.join(extra_list) if extra_list else 'None'}")
        print(f"  - Your Pronunciation Attributes: {', '.join(predicted_attributes)}")
        print(f"  - Expected Attributes: {', '.join(expected_attributes)}\n")
        detail_checking(reference, recognized_phoneme)

    return recognized_phoneme, reference

In [ ]:
# Initialize counters and storage for sentences
i = 0
result = {}

TA_list = []
FA_list = []
TR_list = []
FR_list = []
CD_list = []
WD_list = []

FAR_list = []
FRR_list = []
DER_list = []

total_TA = 0  # True Accept
total_FA = 0  # False Accept
total_TR = 0  # True Reject
total_FR = 0  # False Reject
total_CD = 0  # Correct Detection
total_WD = 0  # Wrong Detection


print("=" * 20 + " The Evaluation stat now " + "=" * 20)
print("=" * 65 + "\n")
for sentence in prediction:
    actual = []
    pred = []
    metrics = []

    # Loop through each item list in the sentence
    for j, item_list in enumerate(sentence):
        # Filter out items containing 'n_' from predicted attributes
        predicted_attributes = [item for item in item_list if "n_" not in item]

        out, ref = check_prediction(predicted_attributes, actual_sentence[i][j], choice, True)

        # Append the results
        actual.append(ref)
        pred.append(out)

        # Stop if reached end of reference for this sentence
        if j >= len(actual_sentence[i]) - 1:
            break

    # Display results for the current sentence
    actual = " ".join(actual)
    pred = " ".join(pred)

    # Case 1) Not equal number of strings evaluations       Levenshtein
    if len(pred.split()) != len(supposed_sentence[i].split()):
        supposed = supposed_sentence[i].split()
        pred = pred.split()
        actual = actual.split()

        print("=" * 20 + " SUMMARY: Not one-to-one " + "=" * 20)
        print(f"Supposed ({len(supposed)}):                           {' '.join(supposed)}")
        print(f"Actual ({len(actual)}):                             {' '.join(actual)}")
        print(f"Model Predictions ({len(pred)}):                  {' '.join(pred)} \n")

        print("Now we will start non one-to-one metrics computation:")  ## !!!!!!!!!!!!!!!!!!!!!
        if len(pred) < len(supposed_sentence[i].split()):

            cm, errors = compute_levenshtein_metrics(supposed, pred)
            cm_2, errors_2 = compute_levenshtein_metrics(supposed, actual)

            aligned_pred, aligned_supposed = align_sentence(pred, supposed, errors)
            aligned_actual, temp = align_sentence(actual, supposed, errors_2)

            cm_3, errors_3 = compute_levenshtein_metrics(aligned_supposed.split(), aligned_actual.split())
            aligned_actual, temp = align_sentence(aligned_actual.split(), aligned_supposed.split(), errors_3)

            print(f"Mandarin sentence(speaker said):          {Mandarin_sentence[i]}")
            print(f"Supposed ({len(aligned_supposed.split())}):                       {aligned_supposed}")
            print(f"Aligned Actual ({len(aligned_actual.split())}):                 {(aligned_actual)}")
            print(f"Aligned Predictions ({len(aligned_pred.split())}):            {(aligned_pred)} \n")

            metrics, diagnosed = compute_metrics(aligned_pred, aligned_actual, aligned_supposed)
            TA, TR, FA, FR, CD, WD, FAR, FRR, DER = compute_metrics_percentage(metrics, diagnosed)

        else:
            cm, errors = compute_levenshtein_metrics(supposed, actual)
            temp, aligned_supposed = align_sentence(actual, supposed, errors)

            print(f"Mandarin sentence(speaker said):          {Mandarin_sentence[i]}")
            print(f"Aligned Supposed ({len(aligned_supposed.split())}):                   {aligned_supposed}")
            print(f"Actual ({len(actual)}):                             {' '.join(actual)}")
            print(f"Model Predictions ({len(pred)}):                  {' '.join(pred)} \n")

            metrics, diagnosed = compute_metrics(" ".join(pred), " ".join(actual), aligned_supposed)
            TA, TR, FA, FR, CD, WD, FAR, FRR, DER = compute_metrics_percentage(metrics, diagnosed)

    else:
        # Case 2) One-to-One matching evaluations
        metrics, diagnosed = compute_metrics(pred, actual, supposed_sentence[i])
        TA, TR, FA, FR, CD, WD, FAR, FRR, DER = compute_metrics_percentage(metrics, diagnosed)
        errors = compute_error_type(pred, supposed_sentence[i])

        print("=" * 20 + " SUMMARY: one-to-one " + "=" * 20)
        print(f"Mandarin sentence(speaker said):          {Mandarin_sentence[i]}")
        print(f"Supposed:                           {supposed_sentence[i]}")
        print(f"Actual:                             {actual}")
        print(f"Model Predictions:                  {pred}")

    print(f"MDD metrics:                        {metrics}")
    print(f"Diagonsed result:                   {diagnosed}")
    print(f"MDD metrics percentage:             TA {TA:.2f}%,  TR {TR:.2f}%,  FA {FA:.2f}%,  FR {FR:.2f}%,  CD {CD:.2f}%,  WD {WD:.2f}%")
    print(f"MDD metrics (detailed)              FAR {FAR:.2f}%,  FRR {FRR:.2f}%,  DER {DER:.2f}%")
    print(f"MDD Error Types:                    {errors}")
    print("=" * 50 + "\n")

    total_TA += TA
    total_FA += FA
    total_TR += TR
    total_FR += FR
    total_CD += CD
    total_WD += WD

    TA_list.append(TA)
    FA_list.append(FA)
    TR_list.append(TR)
    FR_list.append(FR)
    CD_list.append(CD)
    WD_list.append(WD)

    FAR_list.append(FAR)
    FRR_list.append(FRR)
    DER_list.append(DER)

    i += 1

In [ ]:
# Print the total calculations
num_sentences = i
avg_TA = total_TA / num_sentences
avg_FA = total_FA / num_sentences
avg_TR = total_TR / num_sentences
avg_FR = total_FR / num_sentences
avg_CD = total_CD / num_sentences
avg_WD = total_WD / num_sentences

# Compute the three rates
FAR = sum(FAR_list) / num_sentences
FRR = sum(FRR_list) / num_sentences
DER = sum(DER_list) / num_sentences

# Print results
print("=" * 30 + " PERFORMANCE METRICS " + "=" * 30)
print(f"False Acceptance Rate (FAR): {FAR:.2f}")
print(f"False Rejection Rate (FRR):  {FRR:.2f}")
print(f"Diagnostic Error Rate (DER): {DER:.2f}")
print("=" * 80 + "\n")

## Final Confusion Amtrix
print("=" * 30 + " FINAL CONFUSION MATRIX " + "=" * 30)
print(f"Total Sentences Evaluated: {num_sentences}")
print(f"Total MDD Metrics:   TA {total_TA}, TR {total_TR}, FA {total_FA}, FR {total_FR}, CD {total_CD}, WD {total_WD}")
print(f"Average MDD %:       TA {avg_TA:.2f}%, TR {avg_TR:.2f}%, FA {avg_FA:.2f}%, FR {avg_FR:.2f}%, CD {avg_CD:.2f}%, WD {avg_WD:.2f}%")
print("=" * 80 + "\n")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

total_TA = int(total_TA)
total_FA = int(total_FA)
total_TR = int(total_TR)
total_FR = int(total_FR)

conf_matrix = np.array([[total_TA, total_FA], [total_FR, total_TR]])  # TA (True Accept), FA (False Accept)  # FR (False Reject), TR (True Reject)

labels = ["Accept", "Reject"]

plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)

plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix - Phoneme and Tone Recognition")
plt.show()